In [1]:
import sys
import maboss
import pandas as pd
import numpy as np
from pathlib import Path
sys.path.append("/Users/emilieyu/endotehelial-masboss")
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d, RegularGridInterpolator

from src.config import load_sim_config, load_sweep_config, load_spatial_config
from src.paths import *

sim_cfg = load_sim_config()
sweep_cfg = load_sweep_config()
spatial_cfg = load_spatial_config()

from abm.flow_field import FlowField
from abm.endothelial_cell import EndothelialCell
from abm.membrane_node import MembraneNode
from abm.rho_lookup_table import RhoLookupTable
from abm.spring import Spring
from abm.spring_v2 import Spring2

ipylab module is not installed, menus and toolbar are disabled.


In [2]:
LUT = RhoLookupTable(cfg=spatial_cfg, recruitment_dir=BM_RESULTS_DIR)
cell1 = EndothelialCell(cell_id=0, centroid=np.array([0, 0]), lut=LUT, cfg=spatial_cfg)

cell1.nodes

>>> DEBUG: Successfully loaded recruitment parameter sweep data.
>>> DEBUG: Successfully built interpolators
LUT ready | rest: RhoA=0.463 RhoC=0.437
DEBUG: Initialised Cell 0: 
id=0 | n_nodes=12 | centroid=[-0.  0.] 
 target_area=300.000) | current_area=300.000


[MembraneNode(id=0 | pos=[10.  0.] | force=[0. 0.],
 MembraneNode(id=1 | pos=[8.66 5.  ] | force=[0. 0.],
 MembraneNode(id=2 | pos=[5.   8.66] | force=[0. 0.],
 MembraneNode(id=3 | pos=[ 0. 10.] | force=[0. 0.],
 MembraneNode(id=4 | pos=[-5.    8.66] | force=[0. 0.],
 MembraneNode(id=5 | pos=[-8.66  5.  ] | force=[0. 0.],
 MembraneNode(id=6 | pos=[-10.   0.] | force=[0. 0.],
 MembraneNode(id=7 | pos=[-8.66 -5.  ] | force=[0. 0.],
 MembraneNode(id=8 | pos=[-5.   -8.66] | force=[0. 0.],
 MembraneNode(id=9 | pos=[ -0. -10.] | force=[0. 0.],
 MembraneNode(id=10 | pos=[ 5.   -8.66] | force=[0. 0.],
 MembraneNode(id=11 | pos=[ 8.66 -5.  ] | force=[0. 0.]]

## SINGLE SPRING PIPELINE

STRESS FIBRE PIPELINE: Shear flow → lateral springs stretch (L_current > L_cortex) → high tension on aligned junctions → JCAD and TJP1 recruit → RhoC activates → ROCK/FHOD3 drive stress fibre assembly along the flow axis → fibres have a shorter natural length than the current junction length → L_sf shrinks below L_current → fibre tension pulls the lateral nodes inward toward each other → the cell narrows laterally → cytoplasmic pressure, which resists area loss, pushes the end nodes outward → cell elongates along the flow axis.

In [3]:
node_1 = cell1.nodes[0]
node_2 = cell1.nodes[1]
node_1.pos *= 1.05
node_2.pos *= 1.05
spring = Spring2(1, node_1, node_2, 5.176, 1.0, LUT, spatial_cfg)
spring

Spring(id=1 | L_current=5.176 L_cortex=5.176 L_sf=5.176
 k_active=1.000 | align=0.00 | T=0.0000 [cortex=0.0000 sf=0.0000] | 
 DSP=0.000 TJP1=0.000) JCAD=0.000 | RhoA=0.000 RhoC=0.000)

In [4]:
spring.update_geometry(flow_direction=np.array([1.0, 0.0]))
spring

Spring(id=1 | L_current=5.435 L_cortex=5.176 L_sf=5.176
 k_active=1.000 | align=0.26 | T=0.2860 [cortex=0.2592 sf=0.0268] | 
 DSP=0.000 TJP1=0.000) JCAD=0.000 | RhoA=0.000 RhoC=0.000)

Correct – L_current is exactly 5% of rest length from initial x-stretch.
SF tension is contributing meningfully to total because this is a lateral spring

In [5]:
spring.update_signalling()
spring

Spring(id=1 | L_current=5.435 L_cortex=5.176 L_sf=5.176
 k_active=1.000 | align=0.26 | T=0.2860 [cortex=0.2592 sf=0.0268] | 
 DSP=0.402 TJP1=0.495) JCAD=0.005 | RhoA=0.488 RhoC=0.520)

DSP and JCAD high on lateral spring, TJP1 near zero because its not upstream facing.
DSP + JCAD amplified state pushes RhoA above rest value. This drives meaningful cortical stiffening. 

In [6]:
spring.update_remodelling(0.1)
spring

Spring(id=1 | L_current=5.435 L_cortex=5.176 L_sf=5.176
 k_active=1.000 | align=0.26 | T=0.2860 [cortex=0.2592 sf=0.0268] | 
 DSP=0.402 TJP1=0.495) JCAD=0.005 | RhoA=0.488 RhoC=0.520)

In [7]:
spring.apply_forces()
spring


Spring(id=1 | L_current=5.435 L_cortex=5.176 L_sf=5.176
 k_active=1.000 | align=0.26 | T=0.2860 [cortex=0.2592 sf=0.0268] | 
 DSP=0.402 TJP1=0.495) JCAD=0.005 | RhoA=0.488 RhoC=0.520)

## REFACTOR: SPRING

### Testing Intermediates

In [ ]:

params = spatial_cfg['hill_params']['DSP']
if params.get('knocked_out', True):
    print("knocked out")

knocked out


In [19]:
diff = np.array([3, 4]) - np.array([2.5, 4.5])
diff, np.linalg.norm(diff)

(array([ 0.5, -0.5]), np.float64(0.7071067811865476))

### Test Update Geometry